## Libraries

In [1]:
import torch
from torch.utils.data import random_split
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

c:\Users\Normal\Documents\Projects\Advanced-Machine-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device

In [2]:
device = 'cpu'

## Data loading

In [3]:
dataset = TUDataset(root='./data/', name='MUTAG').to(device)

In [4]:
# Split into training and validation
rng = torch.Generator().manual_seed(0)
train_dataset, validation_dataset, test_dataset = random_split(dataset, (100, 44, 44), generator=rng)

# Create dataloader for training and validation
train_loader = DataLoader(train_dataset, batch_size=100)
validation_loader = DataLoader(validation_dataset, batch_size=44)
test_loader = DataLoader(test_dataset, batch_size=44)

## Model

In [57]:
class baseline():
    def __init__(self, train_loader, seed):
        #the training data
        self.train_loader = train_loader
        #setting the seed for reproducibility
        self.seed = seed
        #getting empirical distribution of number of nodes in the training set for further sampling
        counts = []#init of list to store counts of number of nodes in each graph
        for batch in train_loader:
            #sanity check for loading the graph to the device (basically there should be always device, so if it not the error will be raised without the batch)
            if device is not None:
                batch = batch.to(device)
            #getting number of nodes in each graph in the batch
            sizes = (batch.ptr[1:] - batch.ptr[:-1]).detach().cpu()
            #appending the counts to the list
            counts.append(sizes)
        self.emp_dist_number_nodes = torch.cat(counts, dim=0).long()
        

    def sample_number_nodes(self):
        #creating a generator with a fixed seed for reproducibility
        generator = torch.Generator().manual_seed(self.seed)
        #"sampling" the index for taking the number of nodes from the empirical distribution
        idx = torch.randint(0, len(self.emp_dist_number_nodes), (1,), generator=generator).item()#sampling an index from the empirical distribution
        #getting the sample y the index
        sample_num_nodes = self.emp_dist_number_nodes[idx].item()#getting the number of nodes corresponding to the sampled index
        #returning the results
        return sample_num_nodes
    
    def compute_r(self, num_nodes):
        #declaring variable to store densities for each graph
        densities = []
        #iterating through graphs in training data
        for batch in self.train_loader:
            #loading the graph to the device (basically sanity check if device is specified)
            if device is not None:
                batch = batch.to(device)
            #iterating through graphs in the batch
            for graph in batch.to_data_list():
                #getting the number of nodes for the current graph ()
                graph_num_nodes = int(graph.num_nodes)
                #checking if currently considered graph matches number of nodes (plus sanity check for graphs with 1 or less nodes which should not exist in the setup)
                if graph_num_nodes != num_nodes or graph_num_nodes <= 1:
                    continue
                #count unique undirected edges so density does not depend on edge storage format
                unique_edges = set()
                for edge in graph.edge_index.t().detach().cpu().tolist():#iterating through edges in the graph
                    #filtering out self loops
                    if edge[0] != edge[1]:
                        sorted_edge = tuple(sorted(edge))#sorting the edge to make it undirected (escaping counting the same edge twice in different directions)
                        unique_edges.add(sorted_edge)
                #calculating r for the current graph
                num_edges = len(unique_edges)
                total_possible_edges = graph_num_nodes * (graph_num_nodes - 1) / 2
                r = num_edges / total_possible_edges
                #appending current density to the list of densities
                densities.append(r)
        #calculating the average density of the graphs with the sampled number of nodes
        r_final = sum(densities) / len(densities)
        #returning the result
        return r_final

    def sample_graph_adjacency(self):
        #getting the number of nodes for the graph to sample
        N = self.sample_number_nodes()#sampling the number of nodes for the graph
        #getting the density for the sampled number of nodes
        r = self.compute_r(N)#computing the density for the sampled number of nodes
        #sampling a matrix of uniform random numbers
        U = torch.rand((N, N))#sampling a matrix of uniform random numbers
        #getting the upper triangle
        upper_triangle = torch.triu((U < r), diagonal=1)#taking the upper triangle of the matrix to make it undirected and to avoid self loops
        #making the adjacency matrix symmetric
        graph_adj = upper_triangle + upper_triangle.t()
        #returning results
        return graph_adj

Getting the sample

In [54]:
#getting object
model = baseline(train_loader, seed=3)
#sampling a graph adjacency matrix
sample_graph = model.sample_graph_adjacency()

Checking if adjacency matrix is produced

In [55]:
print("shape:", sample_graph.shape)
print("is 2D:", sample_graph.ndim == 2)
print("is square:", sample_graph.ndim == 2 and sample_graph.shape[0] == sample_graph.shape[1])
print("symmetric:", torch.equal(sample_graph, sample_graph.t()))
print("zero diagonal:", torch.all(sample_graph.diag() == 0).item())
print("binary:", torch.all((sample_graph == 0) | (sample_graph == 1)).item())

shape: torch.Size([20, 20])
is 2D: True
is square: True
symmetric: True
zero diagonal: True
binary: True
